In [1]:
%pip install sentence-transformers scikit-learn openai python-dotenv numpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [4]:
sentences = [
    "Python is a programming language.",
    "Python is commonly used for software development.",
    "Dogs are friendly animals.",
    "PostgreSQL is a relational database.",
    "I like working with databases."
]

for i, sentence in enumerate(sentences):
    print(i, sentence)

0 Python is a programming language.
1 Python is commonly used for software development.
2 Dogs are friendly animals.
3 PostgreSQL is a relational database.
4 I like working with databases.


In [5]:
embeddings = model.encode(sentences)

print("Number of sentences:", len(embeddings))
print("Embedding shape:", embeddings.shape)

Number of sentences: 5
Embedding shape: (5, 384)


In [6]:
similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print("Similarity:", similarity[0][0])

Similarity: 0.8306849


In [7]:
similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[2]]
)

print("Similarity:", similarity[0][0])

Similarity: 0.10340686


In [8]:
documents = [
    "Python is a popular programming language.",
    "PostgreSQL is a relational database management system.",
    "FastAPI is a Python framework for building APIs.",
    "Docker is used to package applications into containers.",
    "pgvector allows PostgreSQL to store and search vectors."
]

document_embeddings = model.encode(documents)

print("Documents:", len(documents))
print("Embedding shape:", document_embeddings.shape)

Documents: 5
Embedding shape: (5, 384)


In [9]:
def vector_search(query, top_k=3):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append({
            "document": documents[index],
            "similarity": similarities[index]
        })

    return results

In [10]:
query = "How can I build an API using Python?"

results = vector_search(query)

for result in results:
    print("=" * 60)
    print("Similarity:", round(float(result["similarity"]), 4))
    print("Document:", result["document"])

Similarity: 0.6843
Document: FastAPI is a Python framework for building APIs.
Similarity: 0.4981
Document: Python is a popular programming language.
Similarity: 0.0921
Document: Docker is used to package applications into containers.


In [11]:
document = """
DocuChat is a RAG-powered chat application.

Users can upload documents to DocuChat.
The documents are divided into smaller chunks.
Each chunk is converted into an embedding vector.

The embeddings are stored in PostgreSQL using pgvector.
When a user asks a question, the question is also converted
into an embedding.

DocuChat compares the question embedding with document chunk
embeddings and retrieves the most relevant chunks.

The retrieved chunks are provided to an LLM as context.
The LLM uses the context to generate a grounded answer.
"""

In [12]:
chunk_size = 250
overlap = 50

chunks = []

start = 0

while start < len(document):
    end = start + chunk_size
    chunk = document[start:end]

    chunks.append(chunk)

    start += chunk_size - overlap

print("Number of chunks:", len(chunks))

Number of chunks: 3


In [13]:
for i, chunk in enumerate(chunks):
    print("=" * 60)
    print("CHUNK", i)
    print(chunk)

CHUNK 0

DocuChat is a RAG-powered chat application.

Users can upload documents to DocuChat.
The documents are divided into smaller chunks.
Each chunk is converted into an embedding vector.

The embeddings are stored in PostgreSQL using pgvector.
When a use
CHUNK 1
re stored in PostgreSQL using pgvector.
When a user asks a question, the question is also converted
into an embedding.

DocuChat compares the question embedding with document chunk
embeddings and retrieves the most relevant chunks.

The retrieved chu
CHUNK 2
ieves the most relevant chunks.

The retrieved chunks are provided to an LLM as context.
The LLM uses the context to generate a grounded answer.



In [14]:
chunk_embeddings = model.encode(chunks)

print("Chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)

Chunks: 3
Embedding shape: (3, 384)


In [15]:
def retrieve_chunks(question, top_k=3):
    question_embedding = model.encode([question])

    similarities = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append({
            "chunk_id": int(index),
            "chunk": chunks[index],
            "similarity": float(similarities[index])
        })

    return results

In [16]:
question = "Where are the document embeddings stored?"

results = retrieve_chunks(question, top_k=2)

for result in results:
    print("=" * 60)
    print("Chunk ID:", result["chunk_id"])
    print("Similarity:", round(result["similarity"], 4))
    print(result["chunk"])

Chunk ID: 1
Similarity: 0.4922
re stored in PostgreSQL using pgvector.
When a user asks a question, the question is also converted
into an embedding.

DocuChat compares the question embedding with document chunk
embeddings and retrieves the most relevant chunks.

The retrieved chu
Chunk ID: 0
Similarity: 0.4634

DocuChat is a RAG-powered chat application.

Users can upload documents to DocuChat.
The documents are divided into smaller chunks.
Each chunk is converted into an embedding vector.

The embeddings are stored in PostgreSQL using pgvector.
When a use


In [17]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", bool(api_key))

API key loaded: True


In [18]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

In [19]:
def rag_answer(question, top_k=3):

    # Step 1: Retrieve relevant chunks
    results = retrieve_chunks(question, top_k=top_k)

    # Step 2: Combine chunks
    context = "\n\n".join(
        result["chunk"]
        for result in results
    )

    # Step 3: Create prompt
    prompt = f"""
Answer the question using ONLY the provided context.

If the answer cannot be found in the context,
say "I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    # Step 4: Send context + question to LLM
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content, results

In [20]:
question = "Where are the embeddings stored?"

answer, retrieved = rag_answer(question)

print("ANSWER:")
print(answer)

ANSWER:
The embeddings are stored in PostgreSQL using the pgvector extension.


In [21]:
question = "How does DocuChat answer user questions?"

answer, retrieved = rag_answer(question)

print(answer)

DocuChat answers user questions by first converting the user’s question into an embedding vector. It then compares this question embedding with the embedding vectors of the document chunks that are stored in PostgreSQL using pgvector. The most relevant chunks are retrieved and supplied to a large language model (LLM) as context. The LLM uses this context to generate a grounded answer to the user.


In [22]:
def calculator(expression):
    """
    Simple calculator tool.
    """
    try:
        result = eval(expression, {"__builtins__": {}})
        return result
    except Exception as e:
        return f"Error: {e}"

In [23]:
calculator("25 * 4")


100

In [24]:
calculator("(100 + 50) / 5")

30.0

In [25]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Mathematical expression to calculate"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

In [26]:
messages = [
    {
        "role": "user",
        "content": "What is 125 * 8?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

print(message)

ChatCompletionMessage(content='125 × 8 = **1000**', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User asks: "What is 125 * 8?" We should calculate 125 * 8. That equals 1000. Actually 125*8 = 1000. So answer: 1000. Use function? The user just asks straightforward. Could use calculator function, but we can just answer. Probably fine.')


In [31]:
import json

messages = [
    {
        "role": "user",
        "content": "What is 125 * 8?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

print("LLM response received")

if message.tool_calls:
    print("Tool was selected")

    tool_call = message.tool_calls[0]

    print("Tool name:", tool_call.function.name)
    print("Arguments:", tool_call.function.arguments)
else:
    print("No tool was selected")
    print(message.content)

LLM response received
Tool was selected
Tool name: calculator
Arguments: {"expression":"125 * 8"}


In [32]:
if message.tool_calls:

    tool_call = message.tool_calls[0]

    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "calculator":
        tool_result = calculator(arguments["expression"])

    print("Calculator result:", tool_result)

Calculator result: 1000


In [33]:
messages.append(message)

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(tool_result)
})

final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools
)

print("Final answer:")
print(final_response.choices[0].message.content)

Final answer:
125 multiplied by 8 equals **1,000**.


In [34]:
messages = [
    {
        "role": "user",
        "content": "What is PostgreSQL?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

print(response.choices[0].message.content)

**PostgreSQL** is a free, open‑source relational database management system (RDBMS) that is widely used for everything from small single‑user projects to large, high‑traffic web services. It’s often called “Postgres” for short.

### Core Characteristics

| Feature | What It Means |
|---------|---------------|
| **ACID‑compliant** | Guarantees atomicity, consistency, isolation, and durability of transactions. |
| **SQL‑standard** | Implements the SQL:2008 standard with many extensions. |
| **Open‑source** | Released under the PostgreSQL license (BSD‑style), so you can run, modify, and distribute it freely. |
| **Extensible** | Supports custom data types, operators, functions, indexes, and even whole new languages (e.g., PL/pgSQL, PL/Perl, PL/Python). |
| **Advanced features** | Full‑text search, JSON/JSONB support, spatial data (PostGIS), GIS, array types, window functions, CTEs, and more. |
| **Replication & High Availability** | Streaming replication, logical replication, failover wit

In [35]:
def simple_agent(question):

    print("USER QUESTION:")
    print(question)

    # ------------------------------------------------
    # STEP 1 — Retrieve information from documents
    # ------------------------------------------------

    print("\nSTEP 1: Retrieving document information...")

    retrieved = retrieve_chunks(question, top_k=3)

    context = "\n\n".join(
        result["chunk"]
        for result in retrieved
    )

    print("Retrieved", len(retrieved), "chunks")

    # ------------------------------------------------
    # STEP 2 — Ask LLM what to do
    # ------------------------------------------------

    print("\nSTEP 2: Asking LLM...")

    prompt = f"""
You are a helpful assistant.

Use the document context below to answer the user's question.

You also have access to a calculator when mathematical
calculation is necessary.

DOCUMENT CONTEXT:
{context}

USER QUESTION:
{question}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # ------------------------------------------------
    # STEP 3 — Check whether tool is needed
    # ------------------------------------------------

    if message.tool_calls:

        print("\nSTEP 3: LLM selected a tool")

        messages.append(message)

        for tool_call in message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(
                tool_call.function.arguments
            )

            if function_name == "calculator":

                print("Calling calculator...")
                print("Expression:", arguments["expression"])

                result = calculator(
                    arguments["expression"]
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })

        # ------------------------------------------------
        # STEP 4 — LLM produces final answer
        # ------------------------------------------------

        print("\nSTEP 4: Generating final answer...")

        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        return final_response.choices[0].message.content

    else:

        print("\nSTEP 3: No tool required")

        return message.content

In [36]:
question = """
According to the document, what database stores the embeddings?
Also calculate 25 * 8.
"""

answer = simple_agent(question)

print("\nFINAL ANSWER:")
print(answer)

USER QUESTION:

According to the document, what database stores the embeddings?
Also calculate 25 * 8.


STEP 1: Retrieving document information...
Retrieved 3 chunks

STEP 2: Asking LLM...

STEP 3: LLM selected a tool
Calling calculator...
Expression: 25 * 8

STEP 4: Generating final answer...

FINAL ANSWER:
The embeddings are stored in **PostgreSQL** (using the `pgvector` extension).  
And 25 × 8 = **200**.
